# Lesson 6: Question & Answering Over Documents with RAG

Welcome to Lesson 6! Everything we have built so far (prompts, chains, memory, LCEL) assumed the LLM already "knew" the answer from its training data. But what happens when you need to ask questions about **your own private documents** that the model has never seen?

### The Session Goal
Today, we will build a complete **Retrieval-Augmented Generation (RAG)** pipeline. This is the industry-standard pattern for Q&A systems that lets an LLM answer questions grounded in your own data, eliminating hallucinations and ensuring factual accuracy.

### The Core Concepts
1. **The Knowledge Gap**: Why LLMs cannot answer questions about private or recent data.
2. **Document Loading & Splitting**: Preparing raw text for semantic search.
3. **Embeddings & Vector Stores**: Converting text into searchable numerical representations.
4. **The RAG Chain**: Combining retrieval with generation using LCEL to build a production Q&A system.

In [ ]:
!pip install -q langchain-core langchain-openai langchain-community langchain-text-splitters chromadb

import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

## Step 1: The Problem - The LLM Knowledge Gap

Large Language Models are trained on massive public datasets up to a certain cutoff date. This means they fundamentally **cannot** answer questions about:
- Your company's internal documents
- Recently published information
- Private databases, PDFs, or notes

Let's observe this limitation directly. We will ask the model about a fictional company handbook that obviously does not exist in its training data.

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Ask about information the model cannot possibly know
response = model.invoke("What is the vacation policy at Nexora Technologies?")
print("--- Asking About Private Company Data ---")
print(f"AI Response: {response.content}")
# The model will either hallucinate a generic answer or admit it does not know.

--- Testing our First Custom Pipe ---
Starting Input: 10
Final Pipeline Output: 27


---

## Step 2: Creating a Knowledge Base (Document Loading & Splitting)

The RAG solution works in two phases:
1. **Indexing Phase** (done once): Load documents, split them into chunks, embed them into a vector store.
2. **Query Phase** (done per question): Retrieve relevant chunks, pass them to the LLM as context.

### Why Split Documents?
LLMs have limited context windows, and embedding models work best on small, focused passages. We split large documents into overlapping chunks so that:
- Each chunk is small enough to embed accurately
- Overlap ensures we do not lose context at chunk boundaries

Let's simulate a company knowledge base using plain text documents.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Simulate loading documents (in production, you would use DocumentLoaders for PDFs, web pages, etc.)
raw_documents = [
    Document(
        page_content="""Nexora Technologies Employee Handbook - Vacation Policy.
All full-time employees receive 25 days of paid vacation per year. Vacation days
accrue at a rate of 2.08 days per month. Unused vacation days can be carried over
to the next year, up to a maximum of 10 days. Employees must submit vacation
requests at least 2 weeks in advance through the HR portal. Manager approval is
required for any vacation longer than 5 consecutive days.""",
        metadata={"source": "handbook.pdf", "section": "vacation"}
    ),
    Document(
        page_content="""Nexora Technologies Employee Handbook - Remote Work Policy.
Employees may work remotely up to 3 days per week. A stable internet connection
and a dedicated workspace are required. All remote workers must be available on
Slack during core hours (10am-4pm local time). Equipment allowance of $1500 is
provided annually for home office setup. Team leads may require in-office presence
for sprint planning and quarterly reviews.""",
        metadata={"source": "handbook.pdf", "section": "remote_work"}
    ),
    Document(
        page_content="""Nexora Technologies Employee Handbook - Performance Reviews.
Performance reviews are conducted bi-annually in June and December. Each review
includes self-assessment, peer feedback, and a manager evaluation. Ratings use a
5-point scale: Exceptional, Exceeds Expectations, Meets Expectations, Needs
Improvement, and Unsatisfactory. Promotion eligibility requires at least two
consecutive 'Exceeds Expectations' ratings. Bonus payouts are tied directly to
performance review outcomes and company revenue targets.""",
        metadata={"source": "handbook.pdf", "section": "performance"}
    ),
]

# Split documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]
)

chunks = text_splitter.split_documents(raw_documents)

print(f"--- Document Splitting Results ---")
print(f"Original documents: {len(raw_documents)}")
print(f"After splitting:    {len(chunks)} chunks\n")

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} ({len(chunk.page_content)} chars) [{chunk.metadata['section']}]:")
    print(f"  '{chunk.page_content[:80]}...'\n")

--- Running our First AI Chain ---
Resulting Output:
Why do programmers prefer dark mode? Because light attracts bugs!


---

## Step 3: Embeddings and Vector Store (Semantic Search)

Now we need a way to **find** the right chunks when a user asks a question. This is where embeddings come in.

### How It Works:
1. **Embedding**: Convert each text chunk into a numerical vector (a list of numbers) that captures its semantic meaning.
2. **Storage**: Store these vectors in a vector database (we use ChromaDB, a lightweight in-memory option).
3. **Retrieval**: When a question comes in, embed the question too, then find the chunks whose vectors are closest in meaning.

This is fundamentally different from keyword search. The query "How many days off do I get?" will match the vacation policy chunk even though neither "days off" nor "get" appear in that document.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Initialize the embedding model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 2. Create a vector store and index our chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="nexora_handbook"
)

# 3. Create a retriever interface (returns top 2 most relevant chunks)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 4. Test retrieval with a natural language query
print("--- Testing Semantic Retrieval ---")
query = "How many days off do I get per year?"
results = retriever.invoke(query)

print(f"Query: '{query}'\n")
print(f"Retrieved {len(results)} relevant chunks:\n")
for i, doc in enumerate(results):
    print(f"  Result {i+1} [source: {doc.metadata['section']}]:")
    print(f"  {doc.page_content[:120]}...\n")

--- 🔍 Component Structural Inspection ---
Prompt Input Expects:  {'topic': {'title': 'Topic', 'type': 'string'}}
Prompt Output Returns: A Formatted Prompt Object

Model Input Expects:   A List of Message Objects or Prompt Objects
Model Output Returns:  An AIMessage Object

Parser Input Expects:  An AIMessage Object
Parser Output Returns: string

--- 🚀 Overall Chain Schema ---
The Chain Expects:     {'topic': {'title': 'Topic', 'type': 'string'}}
The Chain Returns:     string


---

## Step 4: Building the RAG Chain with LCEL

Now we combine everything into a single LCEL pipeline. This is where Lesson 5's pipe operator becomes essential.

### The RAG Pipeline Flow:
```text
User Question
   |---> Retriever (finds relevant document chunks)
   |---> Format chunks into context string
   |
   v
Prompt Template (injects context + question)
   |---> ChatModel (generates answer grounded in context)
         |---> Output Parser (returns clean string)
```

We use `RunnablePassthrough` to pass the original question through alongside the retrieved context. This is the parallel data flow pattern from LCEL.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

# 1. Define the RAG prompt template
rag_prompt = ChatPromptTemplate.from_template("""Answer the question based ONLY on the following context.
If the context does not contain enough information to answer, say "I don't have enough information to answer that."

Context:
{context}

Question: {question}

Answer:""")

# 2. Helper function to format retrieved documents into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3. Initialize model and parser
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

# 4. Build the RAG chain using LCEL
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | model
    | parser
)

# 5. Test the chain
print("--- RAG Q&A Chain ---\n")

question = "What is the vacation policy at Nexora Technologies?"
answer = rag_chain.invoke(question)
print(f"Q: {question}")
print(f"A: {answer}\n")

--- Running Sequential Chain ---
Final Marketing Result: "Ecolux: Elevate Your Lifestyle, Sustainably."


---

## Step 5: Testing Multiple Questions and Boundary Cases

A robust Q&A system must handle various question types: direct lookups, questions requiring synthesis across chunks, and questions the knowledge base cannot answer. Let's stress-test our chain.

In [ ]:
test_questions = [
    "How many days per week can I work from home?",
    "What rating do I need for a promotion?",
    "Can I carry over unused vacation days?",
    "What is the company's policy on stock options?",  # Not in our knowledge base
]

print("--- Multi-Question RAG Test ---\n")
for q in test_questions:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")
    print("-" * 60 + "\n")


=== START OF SEQUENTIAL CHAIN LOGS ===
[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "industry": "renewable energy"
}
[chain/start] [chain:RunnableSequence > prompt:PromptTemplate] Entering Prompt run with input:
{
  "industry": "renewable energy"
}
[chain/end] [chain:RunnableSequence > prompt:PromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: Generate a unique, catchy one-word name for a startup company in the renewable energy space. Return ONLY the word."
  ]
}
[llm/end] [chain:RunnableSequence > llm:ChatOpenAI] s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "Energize",
        "generation_info": {
          "finish_reason": "stop",
          "logprobs": null
        },
        "type": "ChatGeneration",
        "message": {
          "lc": 1,
          "type": "constructor",
          "id": [
       

---

## Step 6: Adding Source Citations

In production, users need to know **where** an answer came from. We can modify our chain to return both the answer and the source documents used to generate it.

In [ ]:
from langchain_core.runnables import RunnableParallel

# Build a chain that returns both the answer and source documents
rag_chain_with_sources = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(
    answer=lambda x: (
        rag_prompt | model | parser
    ).invoke({"context": format_docs(x["context"]), "question": x["question"]})
)

print("--- RAG with Source Citations ---\n")
question = "How often are performance reviews conducted?"
result = rag_chain_with_sources.invoke(question)

print(f"Q: {question}\n")
print(f"A: {result['answer']}\n")
print("Sources:")
for doc in result["context"]:
    print(f"  - {doc.metadata['source']} (section: {doc.metadata['section']})")

---

## Summary and Key Takeaways

Today we built a complete RAG (Retrieval-Augmented Generation) Q&A system from scratch:

| Component | Purpose |
|-----------|---------|
| **Document Loader** | Ingests raw text into LangChain Document objects |
| **Text Splitter** | Breaks documents into searchable chunks with overlap |
| **Embeddings** | Converts text into numerical vectors for semantic search |
| **Vector Store** | Indexes and retrieves chunks by similarity |
| **RAG Chain** | LCEL pipeline combining retrieval with LLM generation |

### Why RAG Matters
- Eliminates hallucination by grounding answers in real documents
- Works with any private data source (PDFs, databases, wikis, APIs)
- Scales to millions of documents with production vector databases (Pinecone, Weaviate, pgvector)
- Provides source attribution for trust and auditability

### Production Considerations
- **Chunking strategy** dramatically affects retrieval quality. Experiment with chunk sizes.
- **Hybrid search** (combining keyword + semantic) often outperforms pure vector search.
- **Re-ranking** retrieved results before passing to the LLM improves answer accuracy.
- **Conversational RAG** adds memory (Lesson 4) so follow-up questions work naturally.